# 00 Setup and First Capture

This notebook verifies the first things a PyTorch user needs: TorchLens imports, `tl.trace` runs on a tiny model, and the returned `Trace` can summarize what happened. A `Trace` is the top-level audit object for one model execution.

We start with a small CPU-only module so the notebook is fast and deterministic. The model has two modules and one functional op, which is enough to make a useful graph without noise.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(7)


class TinyClassifier(nn.Module):
    """Tiny MLP used for the first TorchLens audit capture."""

    def __init__(self) -> None:
        """Initialize the layers."""
        super().__init__()
        self.hidden = nn.Linear(4, 6)
        self.output = nn.Linear(6, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run one forward pass."""
        hidden = torch.relu(self.hidden(x))
        return self.output(hidden)


model = TinyClassifier().eval()
x = torch.randn(3, 4)
print(f"torchlens version: {tl.__version__}")
print(f"input shape: {tuple(x.shape)}")

`tl.trace(model, x)` runs the model once and records tensor-producing operations. By default this small trace saves activations, so we can inspect both metadata and tensors.

In [ ]:
trace = tl.trace(model, x)
print(type(trace).__name__)
print(f"model class: {trace.model_class_name}")
print(f"number of indexed records: {len(trace)}")
print(f"input layers: {trace.input_layers}")
print(f"output layers: {trace.output_layers}")
print(f"total activation memory: {trace.total_activation_memory_str}")

The `Trace` representation is intentionally compact: it lists the layer order and tells you whether activations were saved. This is the quickest “it worked” check.

In [ ]:
print(trace)

`summary()` gives a table-oriented overview for a user-facing report. Here we show the default summary and then a smaller custom field selection.

In [ ]:
print(trace.summary())
print("\nSelected fields only:\n")
print(trace.summary(fields=["name", "shape", "params"]))

A trace creates an explicit output layer with the model result. In this build, `raw_output` is only populated when raw-output saving is enabled, so the executable default check compares the saved output-layer tensor to a normal PyTorch forward pass.

In [ ]:
with torch.no_grad():
    expected = model(x)

output_layer = trace[trace.output_layers[0]]
print(f"raw_output saved by default: {trace.raw_output is not None}")
print(f"output layer: {output_layer.layer_label}")
print(f"output layer shape: {output_layer.shape}")
print(f"matches normal forward: {torch.allclose(output_layer.out, expected)}")

## 🔧 Sandbox

Change the hidden width, batch size, or activation function below and rerun the cell to see how the trace and summary change.

In [ ]:
# TODO: try changing hidden_width, batch_size, or torch.tanh to torch.relu.
hidden_width = 5
batch_size = 2

sandbox_model = nn.Sequential(
    nn.Linear(4, hidden_width),
    nn.Tanh(),
    nn.Linear(hidden_width, 1),
).eval()
sandbox_x = torch.randn(batch_size, 4)
sandbox_trace = tl.trace(sandbox_model, sandbox_x)
print(sandbox_trace.summary(fields=["name", "shape", "params"]))